# Overall Fonts

##### Docker: https://docs.aws.amazon.com/sagemaker/latest/dg/amazon-sagemaker-toolkits.html
      
##### SageMaker Python SDK: https://sagemaker.readthedocs.io/en/stable/overview.html#train-a-model-with-the-sagemaker-python-sdk

##### Estimator vs Model vs Predictor: https://sagemaker.readthedocs.io/en/stable/overview.html#:~:text=These%20are%3A,types%20against%20a%20SageMaker%20endpoint

    --> 


# Fonts

#### https://sagemaker.readthedocs.io/en/stable/frameworks/sklearn/using_sklearn.html#sagemaker-scikit-learn-model-server
          
#### https://github.com/aws/sagemaker-training-toolkit

#### https://github.com/aws/sagemaker-inference-toolkit

# 0. Imports

In [ ]:
### Let's divide the workload
1. Initialize Boto3 SDK and create S3 bucket. 
2. Upload data in Sagemaker Local Storage. 
3. Data Exploration and Understanding.
4. Split the data into Train/Test CSV File. 
5. Upload data into the S3 Bucket.
6. Create Training Script
7. Train script in-side Sagemaker container. 
8. Store Model Artifacts(model.tar.gz) into the S3 Bucket. 
9. Deploy Sagemaker Endpoint(API) for trained model, and test it. 

In [ ]:
from sagemaker.sklearn.estimator import SKLearn
import numpy as np
from sagemaker import get_execution_role
import sagemaker
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
import datetime
import time
import tarfile
import boto3
import pandas as pd

In [ ]:
sm_boto3 = boto3.client("sagemaker")
sess = sagemaker.Session()
region = sess.boto_session.region_name


## 1. Load local data

In [ ]:
df = pd.read_csv("mob_price_classification_train.csv")

# 2. Explore 

In [ ]:
# ...

# 3. Preprocessing

In [ ]:
# ...

# 4. Upload data to s3 


In [ ]:
bucket = '<S3_BUCKET>' 
sk_prefix = "sagemaker/path"


In [ ]:
trainpath = sess.upload_data(
    path="train-V-1.csv", bucket=bucket, key_prefix=sk_prefix
)

testpath = sess.upload_data(
    path="test-V-1.csv", bucket=bucket, key_prefix=sk_prefix
)

# 5. Training
Inference can also be set in train script 

In [ ]:
## create the script 
%%writefile train.py


#!/usr/bin/env python3
## optinal: first len tells how to execute script 

from sklearn.externals import joblib
#from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score, roc_curve, auc

import argparse
import os
import pandas as pd 
import numpy as np 

## extra libraris
import statsmodels.api as sm
!echo statsmoodels > requirements.txt



#### INFERENCE 

def model_fn(model_dir):
    """ Loads the model """
    return joblib.load(model_dir)


def input_fn(input_data, content_type):
    """ 
    param input_data : raw data passed by user/reques
    param contenty_type: type of data passed
    """

    print("input_fn-----------------------")

    if request_content_type == "csv":
        # to Pandas 
         daily_energy = pd.DataFrame(input_data, columns=['Datatime', "energy")

    elif request_content_type == "application/json":
        input_data = json.loads(input_data)
        # Convert input data to Pandas DataFrame if needed
        daily_energy = pd.DataFrame(input_data)

    else:
        # If the request_content_type is not as expected, raise an exception
        raise ValueError(f"Content type {request_content_type} is not supported")
        return 

    # Preprocess 
    daily_energy['Datetime'] = pd.to_datetime(daily_energy['Datetime'], format='%Y/%m/%d')
    daily_energy.sort_values(by='Datetime', inplace=True)
    daily_energy.reset_index(inplace=True, drop=True)
    daily_energy = daily_energy.resample('D', on='Datetime').sum()
    daily_energy.index.freq = 'D'
    
    exog3 = pd.DataFrame({'Datetime': daily_energy.index})
    exog3 = exog3.set_index(pd.DatetimeIndex(exog3['Datetime'], freq='D'))
    exog3['sin365'] = np.sin(2 * np.pi * exog3.index.dayofyear / 365.25)
    exog3['cos365'] = np.cos(2 * np.pi * exog3.index.dayofyear / 365.25)
    exog3['sin365_2'] = np.sin(4 * np.pi * exog3.index.dayofyear / 365.25)
    exog3['cos365_2'] = np.cos(4 * np.pi * exog3.index.dayofyear / 365.25)
    exog3 = exog3.drop(columns=['Datetime'])

    return daily_energy, exog3

def predict_fn(input_data, model):
    """
    param input_data: returned by the input_fn
    param model: returned by the model_fn
    returns: predictions 
    """
    daily_energy, exog3 = input_data
    predictions = model_fit.predict(daily_energy, exog=exog3)
    processed_data = {"predictions": predictions.tolist()}
    print("predict_fn-----------------------")
    return processed_data


def output_fn(prediction, accept):
    """
    param: predictions returned from predict_fn
    returns: predictions correctly transformed 
    Post-processing function for model predictions. It gets executed after the predict_fn.
    """
    # Check if accept type is JSON
    if accept != "application/json":
        raise ValueError(f"Accept type {accept} is not supported")

    print("output_fn-----------------------")
    return json.dumps(prediction), accept

#### TRAINING
    
if __name__=='__main__':
    parser = argparse.ArgumentParser()

     # Data, model, and output directories
    parser.add_argument("--model-dir", type=str, default=os.environ.get("SM_MODEL_DIR"))
    parser.add_argument("--train", type=str, default=os.environ.get("SM_CHANNEL_TRAIN"))
    parser.add_argument("--test", type=str, default=os.environ.get("SM_CHANNEL_TEST"))
    parser.add_argument("--train-file", type=str, default="daily_energy_train.csv")
    parser.add_argument("--test-file", type=str, default="daily_energy_test.csv")

    args, _ = parser.parse_known_args()

    print("SKLearn Version: ", sklearn.__version__)
    print("Joblib Version: ", joblib.__version__)

    print("[INFO] Reading data \n ")
    train_df = pd.read_csv(os.path.join(args.train, args.train_file))
    test_df = pd.read_csv(os.path.join(args.test, args.test_file))
    
    print("[INFO] Processing data \n ")

    train_df['Datetime'] = pd.to_datetime(daily_energy['Datetime'], format='%Y/%m/%d')
    train_df.sort_values(by='Datetime', inplace=True)
    train_df.reset_index(inplace=True, drop=True)
    train_df = daily_energy.resample('D', on='Datetime').sum()
    train_df.index.freq = 'D'
    
    exog3 = pd.DataFrame({'Datetime': train_df.index})
    exog3 = exog3.set_index(pd.DatetimeIndex(exog3['Datetime'], freq='D'))
    exog3['sin365'] = np.sin(2 * np.pi * exog3.index.dayofyear / 365.25)
    exog3['cos365'] = np.cos(2 * np.pi * exog3.index.dayofyear / 365.25)
    exog3['sin365_2'] = np.sin(4 * np.pi * exog3.index.dayofyear / 365.25)
    exog3['cos365_2'] = np.cos(4 * np.pi * exog3.index.dayofyear / 365.25)
    exog3 = exog3.drop(columns=['Datetime'])

    print("[INFO] Training data \n ")
    
    model = sm.tsa.statespace.SARIMAX(
        train_df, trend='n',
        order=(3,0,0),
        seasonal_order=(0,1,1,7),
        exog=exog3
    )
    model.fit()
    model_path = os.path.join(args.model_dir, "model.joblib")
    joblib.dump(model, model_path)
    print("Model saved at", model_path)

    print("[INFO] Predicting data \n ")
    predictions = model.predict(test_df, exog=exog3)
    # accuracy 
    print('[INFO] Model Accuracy is: ', test_acc)
    

### A) Train 

In [ ]:
! python train.py --model-dir ./ \
                                --train ./ \
                                --test ./ \

In [ ]:
# Create Training Job for future run; Also it some predictors can automatically create a inference script based on this 
sklearn_estimator.latest_training_job.wait(logs="None")
artifact = sm_boto3.describe_training_job(
    TrainingJobName=sklearn_estimator.latest_training_job.name
)["ModelArtifacts"]["S3ModelArtifacts"]

print("Model artifact persisted at " + artifact)

###  B) Train with SageMaker Sklearn Container
Auto saves package to S3, trought provided arguments nts 

In [ ]:
FRAMEWORK_VERSION = "0.23-1"

sklearn_estimator = SKLearn( 
    entry_point="train.py",
    role=get_execution_role(),
    instance_count=1,
    instance_type='ml.m4.xlarge',
    framework_version='1.0-1',
    base_job_name="timeSeries-train",
    dependencies: ["custom_library"],
    #hyperparameters 
    #max_run 
    # args (argparses of training_script)
    #source_dir = absolute, relative, or S3 URI Path to a directory with any other training source code dependencies 
    #dependencies = list or path of dependencies , ex: ["custom_library"],
)

# s3 path
sklearn_estimator.fit({"train": trainpath, "test": testpath}, wait=True)

In [ ]:
# deploy --> inference
predictor = sklearn_estimator.deploy(
    instance_type='ml.m4.xlarge',
    initial_instance_count=1,
    endpoint_name=endpoint_name,
)
response = predictor.predict(data)

###  C) Train on Notebook Instance

In [ ]:
#
#

In [ ]:
#load 
def download_from_s3_as_file(aws_path, fname,bucket="sagemaker-datalab"):    
    s3 = boto3.resource('s3')
    s3_file = aws_path + "/" + fname  
    s3.Object(bucket, s3_file).download_file(fname)
  

# 6. Testing 

###  A) Inference / Deploy Sagemaker Endpoint(API) 

.A) predictor_cls=AnalysisClass -> class created on this notebook 

.B) source_dir: path to package(.tar.gz : contains both script, models, dependencies, ...)

In [ ]:
from sagemaker.sklearn.model import SKLearnModel
from time import gmtime, strftime

model_name = "Custom-sklearn-model-" + strftime("%Y-%m-%d-%H-%M-%S", gmtime())
model = SKLearnModel(
    name =  model_name,
    model_data=artifact,
    role=get_execution_role(),
    framework_version=FRAMEWORK_VERSION,
    predictor_cls=AnalysisClass     # Costumized class for Prediction (ex: serialization from json to pandas) 
)
#model_name = "Custom-sklearn-model-" + strftime("%Y-%m-%d-%H-%M-%S", gmtime())
model = SKLearnModel(
    model_data="s3://bucket/model.tar.gz"
    role=get_execution_role(),
    framework_version=FRAMEWORK_VERSION,
    predictor_cls=AnalysisClass     
)

In [ ]:
endpoint_name = "Custom-sklearn-model-" + strftime("%Y-%m-%d-%H-%M-%S", gmtime())
predictor = model.deploy(
    initial_instance_count=1,
    instance_type="ml.m4.xlarge",
    endpoint_name=endpoint_name,
)

In [ ]:
print(predictor.predict(testX[features][0:2].values.tolist()))

### Delete Endpoint

In [ ]:
sm_boto3.delete_endpoint(EndpointName=endpoint_name)

###  B) Run Batch Transform 

In [ ]:
transform_output_folder = "batch-transform-output"
output_path = "s3://{}/{}".format(sess.default_bucket(), transform_output_folder)

transformer = sklearn_estimator.transformer(
    instance_count=1,
    instance_type="ml.m4.xlarge",
    output_path=output_path,
    assemble_with="Line",
    accept="text/csv",
)